# Task 3 — Tabular / Clinical Data Model (XGBoost)
**COMP41840 AI for Health**  
**Owner:** Liban  
**Goal:** Binary classification — benign vs malignant — from clinical/genomic tabular features


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score
)
from sklearn.impute import SimpleImputer

import xgboost as xgb

sns.set_theme(style='whitegrid')
SEED = 42
DATA_ROOT = Path('../data/raw')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)

## 3.1 — Load & Inspect Clinical Data

In [ ]:
df = pd.read_csv(DATA_ROOT.parent / 'clinical.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Keep only benign and malignant for binary classification
df_binary = df[df['label'].isin(['benign', 'malignant'])].copy()
df_binary['label_enc'] = (df_binary['label'] == 'malignant').astype(int)

print('Class counts:')
print(df_binary['label'].value_counts())

## 3.2 — Feature Engineering & Preprocessing

In [ ]:
# Identify feature columns — adjust after inspecting the actual CSV
TARGET = 'label_enc'
DROP_COLS = ['label', 'patient_id', 'image_path']  # adjust as needed
feature_cols = [c for c in df_binary.columns if c not in DROP_COLS + [TARGET]]

X = df_binary[feature_cols].copy()
y = df_binary[TARGET].values

# Encode categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')

## 3.3 — Train/Val/Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 3.4 — XGBoost Model

In [ ]:
# Compute class weight ratio for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED,
    early_stopping_rounds=20
)

xgb_model.fit(
    X_train_imp, y_train,
    eval_set=[(X_val_imp, y_val)],
    verbose=50
)

## 3.5 — Test Set Evaluation

In [ ]:
y_pred  = xgb_model.predict(X_test_imp)
y_proba = xgb_model.predict_proba(X_test_imp)[:, 1]

print(classification_report(y_test, y_pred, target_names=['benign', 'malignant']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['benign', 'malignant']
).plot(cmap='Blues')
plt.title('Tabular Model — Confusion Matrix')
plt.savefig(RESULTS_DIR / 'figures/tabular_confusion_matrix.png', dpi=150)
plt.show()

# ROC
RocCurveDisplay.from_predictions(y_test, y_proba, name='XGBoost')
plt.title('Tabular Model — ROC Curve')
plt.savefig(RESULTS_DIR / 'figures/tabular_roc.png', dpi=150)
plt.show()

In [ ]:
# Feature importance
fi = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
fi.tail(20).plot(kind='barh', figsize=(8, 7), title='Top 20 XGBoost Feature Importances')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/tabular_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Save probs/preds for fusion
np.save(RESULTS_DIR / 'tabular_test_probs.npy',  y_proba)
np.save(RESULTS_DIR / 'tabular_test_labels.npy', y_test)

tabular_metrics = {
    'auc':       round(roc_auc_score(y_test, y_proba), 4),
    'precision': round(precision_score(y_test, y_pred), 4),
    'recall':    round(recall_score(y_test, y_pred), 4),
    'f1':        round(f1_score(y_test, y_pred), 4),
}

metrics_path = RESULTS_DIR / 'metrics.json'
existing = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
existing['tabular'] = tabular_metrics
metrics_path.write_text(json.dumps(existing, indent=2))
print('Saved:', tabular_metrics)

## 3.6 — Cross-Validation (robustness check)

In [ ]:
X_all = imputer.fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Simpler model for CV speed
cv_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False, eval_metric='logloss', random_state=SEED
)

cv_scores = cross_val_score(cv_model, X_all, y, cv=cv, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')